# Image Classification API — Google Colab

Notebook này hướng dẫn:
1. Cài đặt thư viện
2. Kiểm tra mô hình Hugging Face trực tiếp
3. Chạy FastAPI server trên Colab
4. Dùng Pinggy để tạo public URL
5. Gọi và kiểm thử API

---
**Mô hình:** `google/vit-base-patch16-224`  
**Link:** https://huggingface.co/google/vit-base-patch16-224


## Bước 1 — Cài đặt thư viện

In [ ]:
!pip install fastapi uvicorn transformers torch Pillow requests pydantic nest_asyncio -q
print("Cài đặt xong!")

## Bước 2 — Kiểm tra mô hình Hugging Face

Chạy thử mô hình trực tiếp (không cần API) để xác nhận mô hình hoạt động

In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

print("Đang tải mô hình google/vit-base-patch16-224 ...")
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)
print("Tải mô hình xong!")

In [ ]:
# Thử phân loại ảnh con chó từ URL
url_dog = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url_dog, headers=headers)

# print(response.status_code)  # debug

image = Image.open(BytesIO(response.content)).convert("RGB")

# Hiển thị ảnh
display(image.resize((300, 300)))

# Phân loại
results = classifier(image, top_k=5)
print("\n📊 Kết quả phân loại (ảnh chó):")
for r in results:
    bar = "█" * int(r['score'] * 30)
    print(f"  {r['label']:<30} {bar} {r['score']:.4f}")

In [ ]:
# Thử phân loại ảnh con mèo từ URL
url_cat = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url_cat, headers=headers)
image = Image.open(BytesIO(response.content)).convert("RGB")

# Hiển thị ảnh
display(image.resize((300, 300)))

# Phân loại
results = classifier(image, top_k=5)
print("\n📊 Kết quả phân loại (ảnh mèo):")
for r in results:
    bar = "█" * int(r['score'] * 30)
    print(f"  {r['label']:<30} {bar} {r['score']:.4f}")

## Bước 3 — Viết và chạy FastAPI server

In [ ]:
import subprocess, time, os

main_py = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline
import base64
import requests as req
from io import BytesIO
from PIL import Image

app = FastAPI(title="Image Classification API")

try:
    classifier = pipeline(
        "image-classification",
        model="google/vit-base-patch16-224"
    )
    model_loaded = True
except Exception as e:
    classifier = None
    model_loaded = False

class ImageURLInput(BaseModel):
    url: str
    top_k: int = 5

class ImageBase64Input(BaseModel):
    image_base64: str
    top_k: int = 5

def run_classifier(image, top_k):
    results = classifier(image, top_k=top_k)
    return [{"label": r["label"], "score": round(r["score"], 4)} for r in results]

@app.get("/")
def root():
    return {
        "name": "Image Classification API",
        "model": "google/vit-base-patch16-224",
    }

@app.get("/health")
def health():
    return {"status": "ok" if model_loaded else "error", "model_loaded": model_loaded}

@app.post("/predict/url")
def predict_from_url(body: ImageURLInput):
    if not body.url.strip():
        raise HTTPException(status_code=400, detail="Truong url khong duoc de trong.")
    if not model_loaded:
        raise HTTPException(status_code=503, detail="Model chua duoc tai.")
    if body.top_k < 1 or body.top_k > 10:
        raise HTTPException(status_code=400, detail="top_k phai tu 1 den 10.")
    try:
        #Thêm User-Agent để không bị chặn
        headers = {"User-Agent": "Mozilla/5.0"}
        response = req.get(body.url, timeout=10, headers=headers)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="Khong the tai anh tu URL.")
    try:
        results = run_classifier(image, body.top_k)
        return {"input_url": body.url, "top_k": body.top_k, "predictions": results}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Loi khi phan loai: {str(e)}")

@app.post("/predict/base64")
def predict_from_base64(body: ImageBase64Input):
    if not body.image_base64.strip():
        raise HTTPException(status_code=400, detail="Truong image_base64 khong duoc de trong.")
    if not model_loaded:
        raise HTTPException(status_code=503, detail="Model chua duoc tai.")
    if body.top_k < 1 or body.top_k > 10:
        raise HTTPException(status_code=400, detail="top_k phai tu 1 den 10.")
    try:
        image_data = base64.b64decode(body.image_base64)
        image = Image.open(BytesIO(image_data)).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="Chuoi base64 khong hop le.")
    try:
        results = run_classifier(image, body.top_k)
        return {"top_k": body.top_k, "predictions": results}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Loi khi phan loai: {str(e)}")
'''

# Ghi file
with open("main.py", "w") as f:
    f.write(main_py)

# Restart server
os.system("pkill -f uvicorn")
time.sleep(1)

proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)

# Kiểm tra
import requests
try:
    r = requests.get("http://localhost:8000/health")
    print("Server OK:", r.json())
except:
    print("Lỗi:", proc.stderr.read().decode())

In [ ]:
# Chạy FastAPI server nền trên Colab bằng nest_asyncio + uvicorn
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_server():
    uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(3)
print("FastAPI server đang chạy tại http://localhost:8000")

In [ ]:
import subprocess
import time
import os

# Dừng process cũ nếu có
os.system("pkill -f uvicorn")
time.sleep(1)

# Chạy uvicorn bằng subprocess
proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app",
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Đợi server khởi động
time.sleep(5)

# Kiểm tra server đã lên chưa
import requests
try:
    r = requests.get("http://localhost:8000/")
    print("Server đang chạy!")
    print(r.json())
except:
    # In log lỗi nếu server chưa lên
    print("Server chưa lên, log lỗi:")
    print(proc.stderr.read().decode())

## Bước 4 — Tạo Public URL bằng Pinggy


In [ ]:
import subprocess, threading, time

# Dừng Pinggy cũ nếu có
os.system("pkill -f 'ssh.*pinggy'")
time.sleep(1)

def run_pinggy():
    proc = subprocess.Popen(
        ["ssh", "-p", "443",
         "-R0:localhost:8000",
         "-o", "StrictHostKeyChecking=no",
         "-o", "ServerAliveInterval=30",
         "qr@a.pinggy.io"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        print(line, end="", flush=True)

t = threading.Thread(target=run_pinggy, daemon=True)
t.start()

# Đợi URL xuất hiện
time.sleep(10)

In [ ]:
import requests
import json

PINGGY_URL = "https://rszih-34-106-37-10.run.pinggy-free.link"

def pretty(response):
    """In JSON đẹp ra màn hình"""
    print(f"Status: {response.status_code}")
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))
    print()

In [ ]:
# ── Test 1: GET / ────────────────────────────────────────
print("=" * 50)
print("TEST 1 — GET /")
print("=" * 50)
r = requests.get(f"{PINGGY_URL}/")
pretty(r)

In [ ]:
# ── Test 2: GET /health ──────────────────────────────────
print("=" * 50)
print("TEST 2 — GET /health")
print("=" * 50)
r = requests.get(f"{PINGGY_URL}/health")
pretty(r)

In [ ]:
# ── Test 3: POST /predict/url — Ảnh con chó ─────────────
print("=" * 50)
print("TEST 3 — POST /predict/url (ảnh con chó)")
print("=" * 50)

r = requests.post(f"{PINGGY_URL}/predict/url", json={
    "url": "https://tse4.mm.bing.net/th/id/OIP.19YReAUFuWTVltkBphY0rQHaD5?rs=1&pid=ImgDetMain&o=7&rm=3",
    "top_k": 3
})
pretty(r)

In [ ]:
# ── Test 4: POST /predict/url — Ảnh con mèo ─────────────
print("=" * 50)
print("TEST 4 — POST /predict/url (ảnh con mèo)")
print("=" * 50)

r = requests.post(f"{PINGGY_URL}/predict/url", json={
    "url": "https://th.bing.com/th/id/R.e6d3675e87e0e79a6308f0c3281cde13?rik=C%2bVch5ryM8ok0w&riu=http%3a%2f%2fclubcatt.com%2fcdn%2fshop%2farticles%2fdomestic-cat-breeds.jpg%3fv%3d1689772643&ehk=D3pifh1ak%2bjKWwuUMQZMoWNejuu%2b2GkaDw7Jy6FzgvE%3d&risl=&pid=ImgRaw&r=0",
    "top_k": 3
})
pretty(r)

In [ ]:
# ── Test 5: POST /predict/base64 — Ảnh tải về encode ────
import requests, base64

print("=" * 50)
print("TEST 5 — POST /predict/base64")
print("=" * 50)

# Thêm User-Agent để không bị Wikipedia chặn
headers = {"User-Agent": "Mozilla/5.0"}
img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg"
img_bytes = requests.get(img_url, headers=headers).content

# Kiểm tra xem có tải đúng ảnh chưa
print(f"Kích thước dữ liệu tải về: {len(img_bytes)} bytes")
print(f"4 byte đầu (JPG hợp lệ phải là: b'\\xff\\xd8\\xff'): {img_bytes[:4]}")

# Encode và gửi lên server
encoded = base64.b64encode(img_bytes).decode("utf-8")

r = requests.post(f"{PINGGY_URL}/predict/base64", json={
    "image_base64": encoded,
    "top_k": 5
})
pretty(r)

In [ ]:
# ── Test 6: Lỗi — URL rỗng ──────────────────────────────
print("=" * 50)
print("TEST 6 — Lỗi: URL rỗng (expect 400)")
print("=" * 50)

r = requests.post(f"{PINGGY_URL}/predict/url", json={"url": ""})
pretty(r)

In [ ]:
# ── Test 7: Lỗi — URL ảnh không tồn tại ─────────────────
print("=" * 50)
print("TEST 7 — Lỗi: URL không hợp lệ (expect 400)")
print("=" * 50)

r = requests.post(f"{PINGGY_URL}/predict/url", json={
    "url": "https://example.com/khong-ton-tai.jpg"
})
pretty(r)

In [ ]:
# ── Test 8: Lỗi — top_k ngoài giới hạn ──────────────────
print("=" * 50)
print("TEST 8 — Lỗi: top_k = 99 (expect 400)")
print("=" * 50)

r = requests.post(f"{PINGGY_URL}/predict/url", json={
    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg",
    "top_k": 99
})
pretty(r)

In [ ]:
# Tạo file test_api.py
test_code = '''
import requests
import base64
import json

BASE = "https://rszih-34-106-37-10.run.pinggy-free.link"  # Thay URL Pinggy của bạn

def pretty(response):
    print(f"Status: {response.status_code}")
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))
    print()

print("=" * 50)

# Test 1: GET /
print("TEST 1 — GET /")
r = requests.get(f"{BASE}/")
pretty(r)

# Test 2: GET /health
print("TEST 2 — GET /health")
r = requests.get(f"{BASE}/health")
pretty(r)

# Test 3: POST /predict/url - ảnh chó (base64)
print("TEST 3 — POST /predict/base64 (ảnh chó)")
headers = {"User-Agent": "Mozilla/5.0"}
img_bytes = requests.get(
    "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg",
    headers=headers
).content
encoded = base64.b64encode(img_bytes).decode("utf-8")
r = requests.post(f"{BASE}/predict/base64", json={"image_base64": encoded, "top_k": 3})
pretty(r)

# Test 4: POST /predict/base64 - ảnh mèo
print("TEST 4 — POST /predict/base64 (ảnh mèo)")
img_bytes = requests.get(
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg",
    headers=headers
).content
encoded = base64.b64encode(img_bytes).decode("utf-8")
r = requests.post(f"{BASE}/predict/base64", json={"image_base64": encoded, "top_k": 3})
pretty(r)

# Test 5: Lỗi - URL rỗng
print("TEST 5 — Lỗi: URL rỗng (expect 400)")
r = requests.post(f"{BASE}/predict/url", json={"url": ""})
pretty(r)

# Test 6: Lỗi - top_k sai
print("TEST 6 — Lỗi: top_k = 99 (expect 400)")
r = requests.post(f"{BASE}/predict/url", json={
    "url": "https://images.dog.ceo/breeds/labrador/n02099712_7003.jpg",
    "top_k": 99
})
pretty(r)
'''

with open("test_api.py", "w", encoding="utf-8") as f:
    f.write(test_code)

print("Đã tạo file test_api.py")

In [ ]:
!python test_api.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')